# Fine-tune the v11 network with pawn structure inputs (Kaggle, GPU)

**Step 2 of 2** (step 1 is notebook 08a, the data). This notebook trains the next network for the engine by **continuing from v11** instead of starting from zero:

- **Same size as now:** 768 piece inputs -> 1024 hidden neurons per side -> 1 output, clipped ReLU. The fast int16 code of the engine works with it.
- **New: 384 pawn structure inputs.** For every square: is there a *passed*, *isolated* or *doubled* pawn, for the own side and for the enemy (3 kinds x 2 sides x 64 squares). The engine computes exactly the same thing (`pawn_structure` in `nnue.rs`). The new weights start at **0**, so at the start this network gives exactly the v11 evaluation and only learns what the pawn inputs add.
- **Data (about 190 million positions):**
  - about 143 million quiet positions from the Lichess evaluation database (Stockfish depth >= 25, the output of notebook 08a), target = the score
  - 50 million positions from Lichess games (notebook 07), target = 70% score + 30% **game result**

## Before you start

1. **Notebook 08a** has run with *Save & Run All* on a CPU session. Its output is the cache of the Lichess evaluation database.
2. A **Kaggle dataset** (*Create* -> *New Dataset*) with two files:
   - `lichess_games.csv` (or `lichess_games.zip` with the csv inside): the positions from Lichess games of notebook 07
   - `nnue_weights.npz`: the v11 weights, from the `models` folder of the project

   Instead of uploading the games you can also add the **output of notebook 07** (`lichess_games.bin`): this notebook reads both formats.

## Run

1. **Add Input**: your dataset, and *Your Work* -> notebook 08a
2. **Settings**: Accelerator **GPU T4 x2** (only one of the two GPUs is used). Internet is not needed.
3. **Save Version** -> **Save & Run All (Commit)**. You can close the browser, the run continues on Kaggle.

The notebook searches all of `/kaggle/input` for the files and prints which files it uses. It trains `num_epochs` epochs and stops in time before the 12 hour limit of a Kaggle session. If it stopped before the last epoch: add **the output of this notebook** as input as well and run it again, it continues after the last finished epoch.

## Output (the *Output* tab of this notebook)

- `nnue.bin`: the engine file (format NNU2 with pawn inputs), ready for the Rust engine
- `nnue_weights.npz`: the weights, to continue training later
- `nnue_finetune.log`, `nnue_finetune.json` and `nnue_finetune_curves.png`
- `checkpoint.pt` and `best_model.pt`: to continue training in a next run

In [ ]:
import json
import os
import shutil
import struct
import time

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from matplotlib import pyplot as plt
from torch import nn

## Settings

In [ ]:
notebook_start = time.time()

input_dir = "/kaggle/input"
out_dir = "/kaggle/working"

wdl_weight = 0.3
use_pawn_features = True

val_size = 2_000_000
test_size = 2_000_000
batch_size = 4096
gpu_data_fraction = 0.6

learning_rate = 0.0003
num_epochs = 6
session_hours = 11.5
reserve_minutes = 20

CP_SCALE = 0.00184104
PIECE_FEATURES = 768
PAWN_FEATURES = 384
NUM_FEATURES = PIECE_FEATURES + PAWN_FEATURES if use_pawn_features else PIECE_FEATURES

log_file = os.path.join(out_dir, "nnue_finetune.log")
checkpoint_file = os.path.join(out_dir, "checkpoint.pt")
best_file = os.path.join(out_dir, "best_model.pt")
engine_path = os.path.join(out_dir, "nnue.bin")
weights_path = os.path.join(out_dir, "nnue_weights.npz")
resume_names = ["checkpoint.pt", "best_model.pt", "nnue.bin", "nnue_weights.npz", "nnue_finetune.log"]
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

print("device:", device, "| GPUs:", torch.cuda.device_count())

## Helpers and input files

In [ ]:
os.makedirs(out_dir, exist_ok=True)


def log(text):
    hours = (time.time() - notebook_start) / 3600
    line = f"[{time.strftime('%H:%M:%S')} | {hours:.2f} h] {text}"
    print(line, flush=True)
    with open(log_file, "a", encoding="utf-8") as f:
        f.write(line + "\n")


def synchronize():
    if device.type == "cuda":
        torch.cuda.synchronize()


def find_files(name):
    return sorted(os.path.join(folder, name) for folder, _, files in os.walk(input_dir) if name in files)


def input_listing():
    return {folder: sorted(files)[:5] for folder, _, files in os.walk(input_dir) if files}


assert device.type == "cuda", "No GPU: choose Accelerator GPU T4 x2 in the settings of the notebook"

checkpoints = find_files("checkpoint.pt")
resume_dir = os.path.dirname(checkpoints[0]) if checkpoints else None
if resume_dir is not None:
    for name in resume_names:
        if os.path.exists(os.path.join(resume_dir, name)):
            shutil.copy(os.path.join(resume_dir, name), out_dir)

log(f"New session on {device}, looking for the input files in {input_dir}...")
games_files = find_files("lichess_games.bin") + find_files("lichess_games.csv") + find_files("lichess_games.zip")
start_weights_files = [path for path in find_files("nnue_weights.npz") if os.path.dirname(path) != resume_dir]
cache_files = find_files("done.json")
assert games_files, f"lichess_games.bin, lichess_games.csv or lichess_games.zip not found. Did you add the dataset with Add Input? Found: {input_listing()}"
assert start_weights_files, f"nnue_weights.npz (the v11 weights) not found. Found: {input_listing()}"
assert cache_files, f"The cache of notebook 08a (lichess_evals_cache/done.json) is not found. Run notebook 08a first and add its output with Add Input. Found: {input_listing()}"

games_file, start_weights_file, cache_dir = games_files[0], start_weights_files[0], os.path.dirname(cache_files[0])
with open(cache_files[0], encoding="utf-8") as f:
    cache_info = json.load(f)
log(f"Games: {games_file} ({os.path.getsize(games_file) / 1e9:.2f} GB)")
log(f"Start weights: {start_weights_file}")
log(f"Lichess evaluation cache: {cache_dir} ({cache_info['positions']:,} positions, depth >= {cache_info['min_depth']})")
if resume_dir is not None:
    log(f"Previous run found in {resume_dir}: training continues from its checkpoint")

## Old data: the cache of notebook 08a

About 143 million quiet positions from the Lichess evaluation database (Stockfish depth >= 25, the best move is not a capture or promotion), already packed by notebook 08a: 32 bytes per board, the side to move and the label for white. `parse_fen` is used for the test positions at the end.

In [ ]:
PIECE_CODES = {"P": 1, "N": 2, "B": 3, "R": 4, "Q": 5, "K": 6, "p": 7, "n": 8, "b": 9, "r": 10, "q": 11, "k": 12}
DIGITS = "12345678"
CACHE_NAMES = ["packed", "side", "labels"]


def parse_fen(fen):
    board_part, color = fen.split(" ")[:2]
    row = np.zeros(64, dtype=np.uint8)
    index = 0
    for char in board_part:
        if char == "/":
            continue
        if char in DIGITS:
            index += int(char)
        else:
            row[index ^ 56] = PIECE_CODES[char]
            index += 1
    return row, color == "w"


def open_cache():
    return tuple(np.load(os.path.join(cache_dir, f"{name}.npy"), mmap_mode="r") for name in CACHE_NAMES)

## New data: positions from Lichess games (notebook 07)

The games file can be in two formats with the same content:

- `lichess_games.bin` (the output of notebook 07): 36 bytes per position: the board (32 bytes, the same packing as the cache), the side to move, the Stockfish score in centipawns for white (mate = ±10000) and the result of the game (0 black won, 1 draw, 2 white won)
- `lichess_games.csv` (also inside a zip file): one position per line with the columns `board` (the first part of a FEN), `turn` (`w` or `b`), `score` (centipawns for white, mate = ±10000) and `result` (1 white won, 0.5 draw, 0 black won)

The csv file is read in parts of 2 million lines. Each part of FEN boards is packed in one go: every digit becomes that many empty squares (`1`), so every board is exactly 64 characters, and a lookup table turns the characters into piece codes.

The target is `(1 - wdl_weight) * score + wdl_weight * result`, both as a number between -1 and 1.

In [ ]:
GAME_RECORD = np.dtype([("board", "u1", 32), ("white", "u1"), ("score", "<i2"), ("result", "u1")])
GAME_COLUMNS = {"board": str, "turn": str, "score": np.int16, "result": np.float32}
FEN_CODES = np.zeros(256, dtype=np.uint8)
for char, piece_code in PIECE_CODES.items():
    FEN_CODES[ord(char)] = piece_code
FEN_SQUARES = np.arange(64) ^ 56


def game_labels(score, result):
    score_labels = 2 / (1 + np.exp(-0.00368208 * score.astype(np.float32))) - 1
    result_labels = 2 * result.astype(np.float32) - 1
    return ((1 - wdl_weight) * score_labels + wdl_weight * result_labels).astype(np.float16)


def pack_boards(boards):
    text = "".join(boards)
    for digit in "2345678":
        text = text.replace(digit, "1" * int(digit))
    chars = np.frombuffer(text.replace("/", "").encode("ascii"), dtype=np.uint8)
    assert len(chars) == 64 * len(boards), "a board in the games file is not a valid FEN board"
    rows = FEN_CODES[chars.reshape(-1, 64)][:, FEN_SQUARES]
    return rows[:, 0::2] | (rows[:, 1::2] << 4)


def load_games(path, limit=None):
    if path.endswith(".bin"):
        records = np.memmap(path, dtype=GAME_RECORD, mode="r")[:limit]
        return np.ascontiguousarray(records["board"]), np.array(records["white"]), game_labels(records["score"], records["result"] / 2)
    parts = []
    for chunk in pd.read_csv(path, dtype=GAME_COLUMNS, chunksize=2_000_000, nrows=limit):
        packed = pack_boards(chunk["board"].tolist())
        white = (chunk["turn"] == "w").to_numpy(np.uint8)
        parts.append((packed, white, game_labels(chunk["score"].to_numpy(), chunk["result"].to_numpy())))
    return tuple(np.concatenate(arrays) for arrays in zip(*parts))

## Features: 768 piece inputs + 384 pawn structure inputs per perspective

Each perspective sees the board from its own side: for black the board is flipped, so "own" and "enemy" always mean the same thing.

The pawn structure is computed on the GPU for a whole batch at once, on an 8x8 grid per board (rank, file):

- **passed:** no enemy pawn in front of it on the same or an adjacent file
- **isolated:** no own pawn on an adjacent file
- **doubled:** another own pawn on the same file

The input number is `768 + kind * 128 + (0 own, 64 enemy) + square`, exactly like `pawn_feature` in the engine.

In [ ]:
PIECE_TYPES = torch.tensor([0, 0, 1, 2, 3, 4, 5, 0, 1, 2, 3, 4, 5], device=device)
PIECE_COLORS = torch.tensor([0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1], device=device)
SQUARES = torch.arange(64, device=device)
PAWN_KINDS = torch.arange(3, device=device).view(1, 3, 1) * 128


def unpack(data_packed):
    low = data_packed & 15
    high = data_packed >> 4
    return torch.stack([low, high], dim=2).reshape(-1, 64).long()


def north(x):
    return (F.pad(x[:, :-1, :], (0, 0, 1, 0)).cumsum(dim=1) > 0).int()


def south(x):
    return north(x.flip(1)).flip(1)


def adjacent(x):
    return F.pad(x[:, :, :-1], (1, 0)) | F.pad(x[:, :, 1:], (0, 1))


def pawn_structure(board):
    pawns = [(board == 1).int().view(-1, 8, 8), (board == 7).int().view(-1, 8, 8)]
    structure = []
    for color in (0, 1):
        own, enemy = pawns[color], pawns[1 - color]
        blockers = south(enemy) if color == 0 else north(enemy)
        files = own | north(own) | south(own)
        passed = own & (1 - (blockers | adjacent(blockers)))
        isolated = own & (1 - adjacent(files))
        doubled = own & (north(own) | south(own))
        structure.append(torch.stack([passed, isolated, doubled], dim=1).view(-1, 3, 64))
    return structure


def perspective_features(board, structure, perspective):
    flipped = SQUARES ^ (56 * perspective)
    own = (PIECE_COLORS[board] != perspective).long() * 384
    pieces = (own + PIECE_TYPES[board] * 64 + flipped).masked_fill(board == 0, NUM_FEATURES)
    if not use_pawn_features:
        return pieces.sort(dim=1).values[:, :32]
    parts = [pieces]
    for color in (0, 1):
        index = PIECE_FEATURES + (0 if color == perspective else 64) + PAWN_KINDS + flipped.view(1, 1, 64)
        parts.append(torch.where(structure[color].bool(), index, NUM_FEATURES).view(-1, 192))
    x = torch.cat(parts, dim=1).sort(dim=1).values
    return x[:, :int((x < NUM_FEATURES).sum(dim=1).max())]


def features(board, white):
    structure = pawn_structure(board) if use_pawn_features else None
    x_white = perspective_features(board, structure, 0)
    x_black = perspective_features(board, structure, 1)
    width = max(x_white.shape[1], x_black.shape[1])
    x_white = F.pad(x_white, (0, width - x_white.shape[1]), value=NUM_FEATURES)
    x_black = F.pad(x_black, (0, width - x_black.shape[1]), value=NUM_FEATURES)
    white = white.bool().unsqueeze(1)
    return torch.where(white, x_white, x_black), torch.where(white, x_black, x_white)

## Model: v11 plus pawn inputs

The same shape as v11: the inputs are added up per perspective (`EmbeddingBag`, one row per input), clipped between 0 and 1, and one linear layer gives the output. During training the output goes through `tanh`, just like the labels; the engine uses the value before `tanh`.

`build_model` copies the v11 weights into the model. The rows of the pawn inputs stay 0.

In [ ]:
class NNUE(nn.Module):

    def __init__(self, num_hidden):
        super().__init__()
        self.feature_transformer = nn.EmbeddingBag(NUM_FEATURES + 1, num_hidden, mode="sum", padding_idx=NUM_FEATURES)
        self.feature_bias = nn.Parameter(torch.zeros(num_hidden))
        self.output = nn.Linear(in_features=2 * num_hidden, out_features=1)

    def forward(self, x_us, x_them):
        us = torch.clamp(self.feature_transformer(x_us) + self.feature_bias, 0, 1)
        them = torch.clamp(self.feature_transformer(x_them) + self.feature_bias, 0, 1)
        x = torch.cat([us, them], dim=1)
        return self.output(x)[:, 0]


def build_model(start):
    weight = start["feature_transformer.weight"]
    assert weight.shape[1] <= NUM_FEATURES, "the start weights have more inputs than this model"
    model = NNUE(num_hidden=weight.shape[0])
    with torch.no_grad():
        model.feature_transformer.weight.zero_()
        model.feature_transformer.weight[:weight.shape[1]] = torch.from_numpy(weight.T.copy())
        model.feature_bias.copy_(torch.from_numpy(start["feature_transformer.bias"]))
        model.output.weight.copy_(torch.from_numpy(start["output.weight"]))
        model.output.bias.copy_(torch.from_numpy(start["output.bias"]))
    return model.to(device)

## Engine file (nnue.bin, format NNU2 with pawn inputs) and a Python version of the engine evaluation

The pawn structure here is a line-by-line copy of the Rust code (bitboards with `north_span`, `south_span` and `adjacent_files`). Before this notebook was made, this Python evaluation was compared with the Rust engine on 5,000 positions from engine matches (4,550 passed, 7,487 isolated and 2,478 doubled pawns): the largest difference was 0.00000.

In [ ]:
FILE_A = 0x0101010101010101
FILE_H = FILE_A << 7
FULL = (1 << 64) - 1


def write_nnue(net, path):
    num_features, num_hidden = net["feature_weights"].shape
    with open(path, "wb") as f:
        f.write(b"NNU2")
        f.write(struct.pack("<5I", 1, 0, int(num_features > PIECE_FEATURES), num_hidden, 0))
        f.write(struct.pack("<64I", *([0] * 64)))
        for part in (net["feature_weights"], net["feature_bias"], np.zeros(num_features), net["output_weight"], net["output_bias"]):
            f.write(np.asarray(part, dtype="<f4").tobytes())


def read_nnue(path):
    with open(path, "rb") as f:
        data = f.read()
    assert data[:4] == b"NNU2", "not an NNU2 file"
    num_buckets, mirror, pawns, num_hidden, num_layer2 = struct.unpack("<5I", data[4:24])
    assert (num_buckets, mirror, num_layer2) == (1, 0, 0), "unexpected NNU2 header"
    num_features = PIECE_FEATURES + PAWN_FEATURES * pawns
    values = np.frombuffer(data, dtype="<f4", offset=24 + 64 * 4)
    sizes = [num_features * num_hidden, num_hidden, num_features, 2 * num_hidden, 1]
    assert sum(sizes) == len(values), "unexpected file size"
    parts = np.split(values, np.cumsum(sizes)[:-1])
    return {
        "feature_weights": parts[0].reshape(num_features, num_hidden),
        "feature_bias": parts[1],
        "output_weight": parts[3].reshape(1, 2 * num_hidden),
        "output_bias": parts[4],
    }


def north_span(squares):
    span = (squares << 8) & FULL
    span |= (span << 8) & FULL
    span |= (span << 16) & FULL
    return span | ((span << 32) & FULL)


def south_span(squares):
    span = squares >> 8
    span |= span >> 8
    span |= span >> 16
    return span | (span >> 32)


def adjacent_files(squares):
    return ((squares << 1) & ~FILE_A & FULL) | ((squares >> 1) & ~FILE_H)


def reference_pawn_structure(row):
    pawns = [0, 0]
    for square in range(64):
        if row[square] == 1:
            pawns[0] |= 1 << square
        elif row[square] == 7:
            pawns[1] |= 1 << square
    structure = []
    for color in (0, 1):
        own, enemy = pawns[color], pawns[1 - color]
        blockers = south_span(enemy) if color == 0 else north_span(enemy)
        files = own | north_span(own) | south_span(own)
        passed = own & ~(blockers | adjacent_files(blockers))
        isolated = own & ~adjacent_files(files)
        doubled = own & (north_span(own) | south_span(own))
        structure.append([passed, isolated, doubled])
    return structure


def reference_features(row, perspective):
    flip = 56 * perspective
    active = []
    for square in range(64):
        code = int(row[square])
        if code == 0:
            continue
        own = 0 if (code <= 6) == (perspective == 0) else 384
        active.append(own + (code - 1) % 6 * 64 + (square ^ flip))
    if use_pawn_features:
        for color, kinds in enumerate(reference_pawn_structure(row)):
            own = 0 if color == perspective else 64
            for kind, squares in enumerate(kinds):
                for square in range(64):
                    if squares >> square & 1:
                        active.append(PIECE_FEATURES + kind * 128 + own + (square ^ flip))
    return active


def reference_eval(net, row, white):
    us, them = (0, 1) if white else (1, 0)
    hidden = []
    for perspective in (0, 1):
        active = reference_features(row, perspective)
        hidden.append(np.clip(net["feature_bias"] + net["feature_weights"][active].sum(axis=0), 0, 1))
    x = np.concatenate([hidden[us], hidden[them]])
    return float((net["output_weight"] @ x + net["output_bias"])[0])

In [ ]:
def model_net(model):
    weights = {name: value.detach().cpu().numpy() for name, value in model.state_dict().items()}
    return {
        "feature_weights": weights["feature_transformer.weight"][:NUM_FEATURES],
        "feature_bias": weights["feature_bias"],
        "output_weight": weights["output.weight"],
        "output_bias": weights["output.bias"],
    }


def start_net(start):
    weight = start["feature_transformer.weight"].T
    feature_weights = np.zeros((NUM_FEATURES, weight.shape[1]), dtype=np.float32)
    feature_weights[:len(weight)] = weight
    return {
        "feature_weights": feature_weights,
        "feature_bias": start["feature_transformer.bias"],
        "output_weight": start["output.weight"],
        "output_bias": start["output.bias"],
    }


def save_weights(model, path):
    net = model_net(model)
    np.savez(
        path,
        num_hidden=np.array(net["feature_bias"].shape[0]),
        num_layer2=np.array(0),
        **{
            "feature_transformer.weight": net["feature_weights"].T,
            "feature_transformer.bias": net["feature_bias"],
            "output.weight": net["output_weight"],
            "output.bias": net["output_bias"],
        },
    )


def export_model(model, path):
    write_nnue(model_net(model), path)


def predict(model, row, white):
    board = torch.from_numpy(np.asarray(row)).long().unsqueeze(0).to(device)
    data_white = torch.tensor([white], device=device)
    model.eval()
    with torch.no_grad():
        x_us, x_them = features(board, data_white)
        return model(x_us, x_them).item()

## Training functions

In [ ]:
loss_fn = nn.MSELoss()


def batches(data, indices, shuffle):
    data_packed, data_side, data_labels = data
    if shuffle:
        indices = indices[torch.randperm(len(indices)).to(indices.device)]
    for start in range(0, len(indices), batch_size):
        batch = indices[start:start + batch_size].long()
        board = unpack(data_packed[batch].to(device, non_blocking=True))
        white = data_side[batch].to(device, non_blocking=True).bool()
        x_us, x_them = features(board, white)
        labels = data_labels[batch].to(device, non_blocking=True).float()
        targets = torch.where(white, labels, -labels)
        yield x_us, x_them, targets


def train_one_epoch(model, optimizer, data, indices):
    model.train()
    total_loss, num_batches = 0., 0

    for x_us, x_them, targets in batches(data, indices, shuffle=True):
        preds = torch.tanh(model(x_us, x_them))
        loss = loss_fn(preds, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.detach()
        num_batches += 1

    return (total_loss / num_batches).item()


def eval_model(model, data, indices):
    model.eval()
    total_loss, num_batches = 0., 0
    total_abs_error, num_preds = 0., 0

    with torch.no_grad():
        for x_us, x_them, targets in batches(data, indices, shuffle=False):
            preds = torch.tanh(model(x_us, x_them))

            total_loss += loss_fn(preds, targets)
            num_batches += 1

            total_abs_error += (preds - targets).abs().sum()
            num_preds += targets.shape[0]

    return (total_loss / num_batches).item(), (total_abs_error / num_preds).item()


def measure_speed(learning_rate, data, indices, num_batches=300, warmup=50):
    model = build_model(start_weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    model.train()

    for i, (x_us, x_them, targets) in enumerate(batches(data, indices, shuffle=True)):
        if i == warmup:
            synchronize()
            start = time.time()
        if i == num_batches:
            break
        loss = loss_fn(torch.tanh(model(x_us, x_them)), targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    synchronize()
    return (num_batches - warmup) * batch_size / (time.time() - start)

## Quick check of the whole pipeline

Fails within minutes if something is wrong:

- the model starts with exactly the v11 evaluation
- the pawn and piece inputs on the GPU are the same as in the Python copy of the engine
- training works, and the exported `nnue.bin` gives the same evaluation as the model
- the games file and the cache of notebook 08a have exactly one king per side in every position

In [ ]:
log("Quick check of the whole pipeline on a small sample...")
start_weights = dict(np.load(start_weights_file))
log(f"v11 start weights: {start_weights['feature_transformer.weight'].shape[1]} inputs -> {start_weights['feature_transformer.weight'].shape[0]} hidden")

sample = tuple(torch.from_numpy(array).to(device) for array in load_games(games_file, limit=20_000))
log(f"First {len(sample[2]):,} positions of {os.path.basename(games_file)}: white to move in {sample[1].float().mean().item():.0%}, mean target for white {sample[2].float().mean().item():+.3f}")
sample_idx = torch.arange(len(sample[2]), device=device, dtype=torch.int32)
board = unpack(sample[0][:1000])
assert bool(((board == 6).sum(dim=1) == 1).all()) and bool(((board == 12).sum(dim=1) == 1).all()), "every position needs exactly one king per side"

check_model = build_model(start_weights)
v11_net = start_net(start_weights)
feature_errors, start_error = 0, 0.
for i in range(300):
    row = unpack(sample[0][i:i + 1])[0].cpu().numpy()
    white = bool(sample[1][i])
    x_us, x_them = features(torch.from_numpy(row).long().unsqueeze(0).to(device), torch.tensor([white], device=device))
    us = 0 if white else 1
    for x, perspective in [(x_us, us), (x_them, 1 - us)]:
        if sorted(index for index in x[0].tolist() if index < NUM_FEATURES) != sorted(reference_features(row, perspective)):
            feature_errors += 1
    start_error = max(start_error, abs(predict(check_model, row, white) - reference_eval(v11_net, row, white)))
log(f"Feature mismatches: {feature_errors}, largest difference between the start model and v11: {start_error:.6f}")
assert feature_errors == 0, "the features differ from the engine"
assert start_error < 1e-4, "the start model is not v11"

check_optimizer = torch.optim.Adam(check_model.parameters(), lr=1e-3)
train_loss = train_one_epoch(check_model, check_optimizer, sample, sample_idx)
val_loss, val_mae = eval_model(check_model, sample, sample_idx)
log(f"One epoch on the sample: train loss {train_loss:.4f}, val loss {val_loss:.4f}, val MAE {val_mae:.4f}")
log(f"Speed: {measure_speed(1e-3, sample, sample_idx, num_batches=4, warmup=1):,.0f} positions/s")

with torch.no_grad():
    for param in check_model.parameters():
        param.add_(0.01 * torch.randn_like(param))
check_path = os.path.join(out_dir, "check.bin")
export_model(check_model, check_path)
check_net = read_nnue(check_path)
os.remove(check_path)
export_error = 0.
for i in range(300):
    row = unpack(sample[0][i:i + 1])[0].cpu().numpy()
    white = bool(sample[1][i])
    export_error = max(export_error, abs(predict(check_model, row, white) - reference_eval(check_net, row, white)))
log(f"Largest difference between the model and the engine file: {export_error:.6f}")
assert export_error < 1e-3, "the engine file does not match the model"

cache = open_cache()
assert len(cache[2]) == cache_info["positions"], "the cache of notebook 08a is not complete"
cache_board = unpack(torch.from_numpy(np.array(cache[0][:1000])).to(device))
assert bool(((cache_board == 6).sum(dim=1) == 1).all()) and bool(((cache_board == 12).sum(dim=1) == 1).all()), "every cached position needs exactly one king per side"
log("Quick check passed")

## Data

The cache of notebook 08a and the games are put together in one dataset. Everything is kept on the GPU if it fits (about 7 GB), so training does not wait for the CPU.

The split into train, validation and test uses a fixed seed, so it is the same in every session.

In [ ]:
log("Loading the cache and the games...")
games = load_games(games_file)
num_lichess, num_games = len(cache[2]), len(games[2])
num_positions = num_lichess + num_games
arrays = tuple(np.concatenate([cached, part]) for cached, part in zip(cache, games))
del cache, games, sample, cache_board

data_gb = sum(array.nbytes for array in arrays) / 1e9
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
data_device = device if data_gb < gpu_data_fraction * gpu_gb else torch.device("cpu")
data = tuple(torch.from_numpy(array).to(data_device) for array in arrays)
del arrays
log(f"Data: {num_positions:,} positions ({num_lichess:,} Lichess evaluations, {num_games:,} from games), {data_gb:.1f} GB, kept on {data_device}")

perm = torch.randperm(num_positions, generator=torch.Generator().manual_seed(42)).int().to(data_device)
val_idx = perm[:val_size]
test_idx = perm[val_size:val_size + test_size]
train_idx = perm[val_size + test_size:]
del perm
log(f"Train: {len(train_idx):,}, val: {len(val_idx):,}, test: {len(test_idx):,}")
log(f"GPU memory in use: {torch.cuda.memory_allocated() / 1e9:.1f} GB of {gpu_gb:.1f} GB")

## Training (continues from the checkpoint if there is one)

Adam with a cosine learning rate schedule over `num_epochs` epochs. The learning rate is lower than when v11 was trained from zero, because the network only has to adjust, not learn chess from the start.

After every epoch the checkpoint is saved in the output folder. When the validation loss is the best so far, `nnue.bin` and `nnue_weights.npz` are saved as well.

In [ ]:
deadline = notebook_start + session_hours * 3600 - reserve_minutes * 60
speed = measure_speed(learning_rate, data, train_idx)
epoch_seconds = 1.1 * len(train_idx) / speed
log(f"Speed: {speed:,.0f} positions/s, about {epoch_seconds / 60:.0f} min per epoch, {num_epochs} epochs in total")

model = build_model(start_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=learning_rate / 100)
history = {"train_loss": [], "val_loss": [], "val_mae": [], "learning_rate": []}
best_val_loss = float("inf")

if os.path.exists(checkpoint_file):
    checkpoint = torch.load(checkpoint_file, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    scheduler.load_state_dict(checkpoint["scheduler"])
    history = checkpoint["history"]
    best_val_loss = checkpoint["best_val_loss"]
    log(f"Continuing from the checkpoint after epoch {len(history['train_loss'])}")
else:
    start_val_loss, start_val_mae = eval_model(model, data, val_idx)
    log(f"Before training (v11, pawn inputs at 0): val loss {start_val_loss:.4f}, val MAE {start_val_mae:.4f}")
print(model)

for epoch in range(len(history["train_loss"]), num_epochs):
    if time.time() + epoch_seconds > deadline:
        log("Not enough time left in this Kaggle session for another epoch: add the output of this notebook as input and run it again to continue")
        break

    current_lr = optimizer.param_groups[0]["lr"]
    train_loss = train_one_epoch(model, optimizer, data, train_idx)
    val_loss, val_mae = eval_model(model, data, val_idx)
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_mae"].append(val_mae)
    history["learning_rate"].append(current_lr)
    log(f"Epoch {epoch + 1}/{num_epochs}: train loss {train_loss:.4f}, val loss {val_loss:.4f}, val MAE {val_mae:.4f}, lr {current_lr:.6f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_file)
        save_weights(model, weights_path)
        export_model(model, engine_path)
        log("Best so far: saved nnue.bin and nnue_weights.npz")
    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(), "history": history, "best_val_loss": best_val_loss}, checkpoint_file)

finished = len(history["train_loss"]) == num_epochs
log("Training finished" if finished else f"Training paused after epoch {len(history['train_loss'])} of {num_epochs}")

## Test

The best network so far on the test positions, compared with v11 on the same positions.

In [ ]:
if not finished:
    log("The output has the best network so far. To continue: Add Input -> the output of this notebook, then Save & Run All again")
model.load_state_dict(torch.load(best_file, map_location=device))
test_loss, test_mae = eval_model(model, data, test_idx)
v11_test_loss, v11_test_mae = eval_model(build_model(start_weights), data, test_idx)
log(f"Test: loss {test_loss:.4f}, MAE {test_mae:.4f} (v11: loss {v11_test_loss:.4f}, MAE {v11_test_mae:.4f})")

## Sanity check, piece values and what the pawn inputs learned

The piece values are measured like this: the evaluation of a position where black misses one piece, minus the evaluation of the same position with all pieces. For comparison, v11 gave: opening pawn +64, knight +363, bishop +417, rook +446, queen +667; endgame pawn +165, 2 pawns +368, rook +923.

The pawn part is the evaluation of the network minus the evaluation of the same network with the pawn input weights set to 0.

In [ ]:
test_fens = {
    "start position": "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1",
    "white a bishop down, white to move": "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQK1NR w KQkq - 0 1",
    "white a queen down, white to move": "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNB1KBNR w KQkq - 0 1",
    "rook against 3 pawns, white to move (about +200)": "1nbqkbnr/pppppppp/8/8/8/8/3PPPPP/RNBQKBNR w KQk - 0 1",
    "lost game 1, black a bishop down (Stockfish -639)": "2r3k1/5p1p/bq2p1p1/pN1pP3/P2P2n1/RP1Q1NP1/5PBP/6K1 b - - 0 22",
    "lost game 2, black to move (Stockfish -387)": "2krr3/1pp2p1p/p1nb3q/3p4/Q1PP2pN/3bP1P1/P2N1PBP/R2R2K1 b - - 0 16",
}

engine_net = read_nnue(engine_path)
sanity = {}
for name, fen in test_fens.items():
    row, white = parse_fen(fen)
    raw = predict(model, row, white)
    engine_raw = reference_eval(engine_net, row, white)
    sanity[name] = {"fen": fen, "raw": raw}
    log(f"{name}: {raw / CP_SCALE:+.0f} cp for the side to move (engine file: {engine_raw / CP_SCALE:+.0f} cp)")


material_tests = {
    "opening": ("rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1", {
        "pawn": "rnbqkbnr/1ppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1",
        "knight": "r1bqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1",
        "bishop": "rn1qkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1",
        "rook": "1nbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQk - 0 1",
        "queen": "rnb1kbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1",
    }),
    "endgame": ("r3k3/ppp2ppp/8/8/8/8/PPP2PPP/R3K3 w - - 0 1", {
        "pawn": "r3k3/ppp2pp1/8/8/8/8/PPP2PPP/R3K3 w - - 0 1",
        "2 pawns": "r3k3/ppp2p2/8/8/8/8/PPP2PPP/R3K3 w - - 0 1",
        "rook": "4k3/ppp2ppp/8/8/8/8/PPP2PPP/R3K3 w - - 0 1",
    }),
}

piece_values = {}
for phase, (start_fen, missing) in material_tests.items():
    start_value = predict(model, *parse_fen(start_fen))
    for name, fen in missing.items():
        value = (predict(model, *parse_fen(fen)) - start_value) / CP_SCALE
        piece_values[f"{phase}: black without {name}"] = value
        log(f"{phase}, black without {name}: {value:+.0f} cp for white")

pawn_tests = {
    "white passed pawn on d6, white to move": "6k1/5ppp/3P4/8/8/8/5PPP/6K1 w - - 0 1",
    "black isolated pawn on d5, black to move": "r1bq1rk1/pp3ppp/2n2n2/3p4/3N4/2N1B3/PP2BPPP/R2Q1RK1 b - - 0 10",
    "white doubled c-pawns, black to move": "r1bqk2r/pppp1ppp/2n2n2/4p3/4P3/2P2N2/P1PP1PPP/R1BQKB1R b KQkq - 0 5",
}
pawn_effects = {}
if use_pawn_features:
    no_pawn_model = build_model(start_weights)
    no_pawn_model.load_state_dict(model.state_dict())
    with torch.no_grad():
        no_pawn_model.feature_transformer.weight[PIECE_FEATURES:NUM_FEATURES] = 0
    for name, fen in pawn_tests.items():
        row, white = parse_fen(fen)
        total, without = predict(model, row, white), predict(no_pawn_model, row, white)
        pawn_effects[name] = (total - without) / CP_SCALE
        log(f"{name}: {total / CP_SCALE:+.0f} cp for the side to move, of which the pawn inputs {pawn_effects[name]:+.0f} cp")

params = {
    "num_hidden": model.feature_bias.shape[0],
    "num_features": NUM_FEATURES,
    "use_pawn_features": use_pawn_features,
    "activation": "clipped relu",
    "start": "v11",
    "learning_rate": learning_rate,
    "wdl_weight": wdl_weight,
    "num_lichess": num_lichess,
    "num_games": num_games,
    "min_depth": cache_info["min_depth"],
    "num_epochs": len(history["train_loss"]),
    "positions_per_second": speed,
}
with open(os.path.join(out_dir, "nnue_finetune.json"), "w", encoding="utf-8") as f:
    json.dump({"params": params, "history": history, "test_loss": test_loss, "test_mae": test_mae, "v11_test_loss": v11_test_loss, "v11_test_mae": v11_test_mae, "sanity": sanity, "piece_values": piece_values, "pawn_effects": pawn_effects}, f, indent=2)
log("Saved nnue_finetune.json")

## Plots

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(15, 4))

plt.subplot(1, 3, 1)
plt.plot(epochs, history["train_loss"], label="Train loss")
plt.plot(epochs, history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(epochs, history["val_mae"], label="Validation MAE")
plt.xlabel("Epoch")
plt.ylabel("MAE")
plt.legend()

plt.subplot(1, 3, 3)
plt.plot(epochs, history["learning_rate"], label="Learning rate")
plt.xlabel("Epoch")
plt.ylabel("Learning rate")
plt.yscale("log")
plt.legend()

plt.savefig(os.path.join(out_dir, "nnue_finetune_curves.png"), bbox_inches="tight")
plt.show()

In [ ]:
log(f"Done. Download nnue.bin from the Output tab of this notebook. Files: {sorted(os.listdir(out_dir))}")